# Cross-Check Lineage CSVs with Live CMR `metadataAssociations`

This notebook checks every association in the ATBD-derived lineage CSVs against the `metadataAssociations`
that CMR holds **right now**. The data comes straight from the CMR GraphQL API, not from a saved snapshot.

**Input**
- `lineage/metrics/<PROVIDER>/*_lineage.csv` for `<PROVIDER>` in `ASF`, `GES_DISC`, `GHRC_DAAC`, `LANCEMODIS`,
  `LARC_CLOUD`, `LPCLOUD`, `LPDAAC_ECS`. Each directory name is a CMR provider ID.
- CMR GraphQL: `https://graphql.earthdata.nasa.gov/api`

**Output**
- The same `*_lineage.csv` files, rewritten in place with a new `match` column at the end.
- `lineage/reports/Collection_MetadataAssociations_CrossCheck_Report.md`, generated in Step 9 from this run's results.

**Ground rule:** any `match`/`Match` column already in the CSVs is treated as if it doesn't exist. It is
dropped on read, never used, and replaced by the newly computed `match` column.

**`match` values**

| Value | Meaning |
|---|---|
| `true` | The related collection is in the CMR record's `metadataAssociations` with the **same** type |
| `<TYPE>` (e.g. `INPUT`) | The related collection is in the CMR record but with a **different** type. The CMR type is shown. If CMR lists it under several types, they are joined with `\|` |
| `false` | The related collection is not in the CMR record's `metadataAssociations`, or the source collection isn't in CMR at all |

**Steps**
1. Inventory the CSVs
2. Choose a query strategy (one request per collection vs. batched vs. whole provider) and benchmark it
3. Fetch the collections from CMR GraphQL
4. Resolve each CSV source collection to a CMR record
5. Define the matching rules
6. Compute `match` and write the CSVs
7. Analysis
8. Summary
9. Write the report

In [1]:
import csv
import json
import statistics
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import requests

# ---- Inputs ---------------------------------------------------------------
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lineage" / "metrics").is_dir())
METRICS_DIR = REPO_ROOT / "lineage" / "metrics"
PROVIDERS = ["ASF", "GES_DISC", "GHRC_DAAC", "LANCEMODIS", "LARC_CLOUD", "LPCLOUD", "LPDAAC_ECS"]
GRAPHQL_URL = "https://graphql.earthdata.nasa.gov/api"

# ---- Options --------------------------------------------------------------
DRY_RUN = False          # True = compute and summarize without rewriting any CSV
BATCH_SIZE = 100         # short names per GraphQL request
RUN_BENCHMARK = True     # Step 2: time a sample of one-collection-per-request queries
BENCHMARK_SAMPLE = 10

OLD_MATCH_COLUMNS = {"match", "Match"}   # existing columns to ignore and replace
session = requests.Session()
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/eleu/dev/esdis/metadata-quality-skills


## Step 1: Inventory the lineage CSVs

Only `*_lineage.csv` files directly inside each provider directory are read. The `*_lineage_evidence.json`
files and the CSVs at the top of `lineage/metrics/` (NISAR, SWOT) are not part of this exercise.

The old `match`/`Match` column is dropped right away, so none of the later steps can see it.

In [2]:
def norm(s):
    # Trim and uppercase for comparison. Some CMR entryIds have trailing spaces or different case.
    return (s or "").strip().upper()


files = []   # one dict per CSV: provider, path, fieldnames (without old match), rows
for prov in PROVIDERS:
    for path in sorted((METRICS_DIR / prov).glob("*_lineage.csv")):
        with path.open(newline="") as fh:
            reader = csv.DictReader(fh)
            fieldnames = [c for c in reader.fieldnames if c not in OLD_MATCH_COLUMNS]
            rows = [{k: v for k, v in r.items() if k in fieldnames} for r in reader]
        files.append({"provider": prov, "path": path, "fieldnames": fieldnames, "rows": rows})

sources = {prov: sorted({(r["shortname"].strip(), r["version"].strip())
                         for f in files if f["provider"] == prov for r in f["rows"]})
           for prov in PROVIDERS}

print(f"{'Provider':<11} {'CSV files':>9} {'rows':>6} {'source collections':>19}")
for prov in PROVIDERS:
    fs = [f for f in files if f["provider"] == prov]
    print(f"{prov:<11} {len(fs):>9} {sum(len(f['rows']) for f in fs):>6} {len(sources[prov]):>19}")
print(f"{'ALL':<11} {len(files):>9} {sum(len(f['rows']) for f in files):>6} {sum(map(len, sources.values())):>19}")
print(f"\nColumns: {sorted({tuple(f['fieldnames']) for f in files})}")
print(f"Header-only files: {[str(f['path'].relative_to(METRICS_DIR)) for f in files if not f['rows']]}")

Provider    CSV files   rows  source collections
ASF                 6    267                  41
GES_DISC            9    812                 157
GHRC_DAAC           1      2                   1
LANCEMODIS          4    104                  20
LARC_CLOUD         29   1836                 510
LPCLOUD             1     14                   3
LPDAAC_ECS          1      4                   2
ALL                51   3039                 734

Columns: [('shortname', 'version', 'association_type', 'related_shortname', 'related_version', 'confidence', 'lineage_relationship')]
Header-only files: ['LARC_CLOUD/SCAR_A_lineage.csv']


## Step 2: Choose a query strategy

There are about 730 source collections across 7 providers. Three ways to get their `metadataAssociations` from CMR GraphQL:

| Strategy | Requests | What comes back | Trade-offs |
|---|---|---|---|
| **A. One request per collection** (`provider` + `shortName` + `version`) | ~730 | Exactly the records needed | Slowest: every request has its own round-trip latency. More requests means more exposure to throttling and transient errors, and a heavier load on a shared public service |
| **B. Whole provider** (`provider` only, paged with `cursor`) | A few pages per provider | Every collection of the 7 providers (see counts below) | Few requests, but pulls many records that aren't needed. It's essentially the snapshot approach again |
| **C. Batched by provider + `shortNames` list** (chosen) | ~12 | Only the needed short names (all versions of each) | Few requests and little extra data. Returning every version of a short name also allows a fallback when the CSV version is wrong (Step 4) |

The GraphQL schema allows this: `CollectionsInput.shortNames` and `entryId` are `[String]` lists, while
`shortName` is a single string. Option C uses `shortNames` rather than `entryId`, so that a version mismatch in
the CSV still returns the record.

**Best-practice choices for option C**
- Batch size of 100 short names. That keeps the request URL/body and the response small, and one failed batch is cheap to retry.
- Ask only for the fields needed: `conceptId shortName version provider consortiums metadataAssociations`.
- Page with `cursor` until `count` items are received. CMR returns a cursor even on the last page, so the loop
  stops on the item count, not on the cursor.
- Retry with exponential backoff. During development one batch failed with a transient error and worked on retry.
- Reuse one HTTP session, and record the retrieval time so the result can be dated.

The cell below backs up the choice with measurements: the latency of a sample of single-collection
requests (option A), and the number of records each provider holds (what option B would fetch).

In [3]:
QUERY = """
query Collections($params: CollectionsInput) {
  collections(params: $params) {
    count
    cursor
    items { conceptId shortName version provider consortiums metadataAssociations }
  }
}
"""


def post_graphql(params, retries=5):
    """Return the `collections` block. Retries with exponential backoff and raises if every attempt fails."""
    err = None
    for attempt in range(retries):
        try:
            r = session.post(GRAPHQL_URL, timeout=120, json={"query": QUERY, "variables": {"params": params}})
            body = r.json()
            data = (body.get("data") or {}).get("collections")
            if r.status_code == 200 and data is not None:
                return data
            err = f"HTTP {r.status_code}: {(body.get('errors') or [{}])[0].get('message')}"
        except (requests.RequestException, ValueError) as e:
            err = repr(e)
        time.sleep(2 ** attempt)
    raise RuntimeError(f"GraphQL failed after {retries} attempts for {params}: {err}")


def fetch_all(params, page_size=2000):
    """Fetch every item that matches `params`, paging with `cursor` until `count` items are received."""
    items, p = [], {**params, "limit": page_size}
    while True:
        data = post_graphql(p)
        items.extend(data["items"] or [])
        if len(items) >= data["count"] or not data["items"]:
            return items
        p = {**params, "limit": page_size, "cursor": data["cursor"]}


if RUN_BENCHMARK:
    sample = [(prov, *s) for prov in PROVIDERS for s in sources[prov]][::73][:BENCHMARK_SAMPLE]
    latencies = []
    for prov, name, ver in sample:
        t0 = time.perf_counter()
        post_graphql({"provider": prov, "shortName": name, "version": ver, "limit": 10})
        latencies.append(time.perf_counter() - t0)
    n_sources = sum(map(len, sources.values()))
    mean = statistics.mean(latencies)
    print(f"Option A sample: {len(latencies)} single-collection requests, mean {mean:.2f} s, "
          f"max {max(latencies):.2f} s")
    print(f"  -> about {n_sources} requests, roughly {n_sources * mean / 60:.1f} min if run one after another\n")

    print("Option B: records per provider (what a whole-provider fetch would download)")
    totals = {prov: post_graphql({"provider": prov, "limit": 0})["count"] for prov in PROVIDERS}
    for prov in PROVIDERS:
        print(f"  {prov:<11} {totals[prov]:>6,} records, {len(sources[prov]):>4} needed")
    print(f"  {'ALL':<11} {sum(totals.values()):>6,} records, {n_sources:>4} needed")

Option A sample: 10 single-collection requests, mean 0.45 s, max 0.73 s
  -> about 734 requests, roughly 5.5 min if run one after another

Option B: records per provider (what a whole-provider fetch would download)


  ASF            177 records,   41 needed
  GES_DISC     2,039 records,  157 needed
  GHRC_DAAC      703 records,    1 needed
  LANCEMODIS     142 records,   20 needed
  LARC_CLOUD   1,749 records,  510 needed
  LPCLOUD        798 records,    3 needed
  LPDAAC_ECS       2 records,    2 needed
  ALL          5,610 records,  734 needed


## Step 3: Fetch the collections from CMR (option C)

One request per (provider, batch of up to 100 short names). Every version of each short name is returned.
The retrieval time is recorded so the results can be dated.

In [4]:
t0 = time.time()
cmr_items, n_requests = [], 0
for prov in PROVIDERS:
    names = sorted({name for name, _ in sources[prov]})
    for i in range(0, len(names), BATCH_SIZE):
        batch = fetch_all({"provider": prov, "shortNames": names[i:i + BATCH_SIZE]})
        n_requests += 1
        cmr_items.extend(batch)
retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
fetch_seconds = time.time() - t0

cmr_items = list({c["conceptId"]: c for c in cmr_items}.values())   # de-duplicate defensively
print(f"Retrieved {len(cmr_items)} CMR records with {n_requests} batched requests in {fetch_seconds:.1f} s")
print(f"Retrieved at (UTC): {retrieved_at}")
print(f"Records per provider: {dict(Counter(c['provider'] for c in cmr_items))}")
print(f"All records in the EOSDIS consortium: {all('EOSDIS' in (c['consortiums'] or []) for c in cmr_items)}")
print(f"Records with metadataAssociations: {sum(bool(c['metadataAssociations']) for c in cmr_items)}")

Retrieved 787 CMR records with 12 batched requests in 11.2 s
Retrieved at (UTC): 2026-11-24T22:35:09+00:00
Records per provider: {'ASF': 41, 'GES_DISC': 187, 'GHRC_DAAC': 1, 'LANCEMODIS': 16, 'LARC_CLOUD': 537, 'LPCLOUD': 3, 'LPDAAC_ECS': 2}
All records in the EOSDIS consortium: True
Records with metadataAssociations: 740


## Step 4: Resolve each CSV source collection to a CMR record

1. **Exact:** same provider, short name and version.
2. **Single-version fallback:** if the exact version isn't in CMR but the provider has exactly one version of that
   short name, use that record. There's only one candidate, so there's no ambiguity, and the CSV version is
   probably wrong. These cases are listed so they can be reviewed.
3. **Not found:** otherwise. Every row for that source gets `match = false`.

In [5]:
cmr_by_key, cmr_by_name = {}, defaultdict(list)
for c in cmr_items:
    cmr_by_key[(c["provider"], norm(c["shortName"]), (c["version"] or "").strip())] = c
    cmr_by_name[(c["provider"], norm(c["shortName"]))].append(c)


def resolve(prov, shortname, version):
    """Return (record or None, how it was resolved)."""
    rec = cmr_by_key.get((prov, norm(shortname), version.strip()))
    if rec:
        return rec, "exact"
    candidates = cmr_by_name.get((prov, norm(shortname)), [])
    if len(candidates) == 1:
        return candidates[0], "single_version_fallback"
    return None, "not_in_cmr" if not candidates else "version_ambiguous"


resolution = {(prov, *s): resolve(prov, *s) for prov in PROVIDERS for s in sources[prov]}
print(Counter(how for _, how in resolution.values()))
for (prov, name, ver), (rec, how) in sorted(resolution.items()):
    if how != "exact":
        found = f"-> CMR {rec['conceptId']} version {rec['version']}" if rec else ""
        print(f"  {how:<24} {prov:<11} {name:<20} csv version {ver:<6} {found}")

Counter({'exact': 730, 'not_in_cmr': 2, 'single_version_fallback': 2})
  not_in_cmr               LANCEMODIS  VNP14IMGTDL_NRT      csv version 2      
  single_version_fallback  LANCEMODIS  VNP46A1G_NRT         csv version 2      -> CMR C1990149134-LANCEMODIS version 1
  single_version_fallback  LANCEMODIS  VNP46A1_NRT          csv version 2      -> CMR C1989175258-LANCEMODIS version 1
  not_in_cmr               LANCEMODIS  VNP46A2_NRT          csv version 2      


## Step 5: Define the matching rules

For one CSV row and its resolved CMR record:

1. Collect the record's `metadataAssociations` entries whose `entryId` equals `related_shortname`. Case and
   surrounding spaces are ignored.
2. **Version as a tie-breaker:** if any of those entries has a `version` equal to the CSV `related_version`, keep only
   those. This matters when a record links to several versions of the same short name with different types (e.g.
   CERES `PARENT` = preceding version, `CHILD` = succeeding version). Otherwise all entries are kept, because CMR
   often leaves the association `version` null.
3. Same `association_type` gives `true`. A different type gives the CMR type(s). No entry gives `false`.

In [6]:
def match_value(rec, row):
    if rec is None:
        return "false"
    related = norm(row["related_shortname"])
    entries = [a for a in rec.get("metadataAssociations") or [] if norm(a.get("entryId")) == related]
    if not entries:
        return "false"
    same_ver = [a for a in entries if (a.get("version") or "").strip() == row["related_version"].strip()]
    entries = same_ver or entries
    db_types = list(dict.fromkeys((a.get("type") or "").strip() or "(no type)" for a in entries))
    if norm(row["association_type"]) in {norm(t) for t in db_types}:
        return "true"
    return "|".join(db_types)


# Sanity check with known examples
for prov, src, ver, rel, rel_ver, typ in [
    ("ASF", "OPERA_L2_CSLC-S1_V1", "1", "SENTINEL-1A_SLC", "1", "PARENT"),
    ("ASF", "OPERA_L2_CSLC-S1_V1", "1", "SENTINEL-1A_SLC", "1", "RELATED"),
    ("ASF", "OPERA_L2_CSLC-S1_V1", "1", "NOT_A_COLLECTION", "1", "PARENT"),
]:
    rec, _ = resolve(prov, src, ver)
    row = {"related_shortname": rel, "related_version": rel_ver, "association_type": typ}
    print(f"{src} -> {rel:<17} csv={typ:<8} match={match_value(rec, row)}")

OPERA_L2_CSLC-S1_V1 -> SENTINEL-1A_SLC   csv=PARENT   match=INPUT
OPERA_L2_CSLC-S1_V1 -> SENTINEL-1A_SLC   csv=RELATED  match=INPUT
OPERA_L2_CSLC-S1_V1 -> NOT_A_COLLECTION  csv=PARENT   match=false


## Step 6: Compute `match` and write the CSVs

Each file keeps its original columns and row order. The old `match`/`Match` column is removed, and the new `match`
column is appended last. Files are written with `\n` line endings, like the originals.

In [7]:
all_rows = []
for f in files:
    for row in f["rows"]:
        key = (f["provider"], row["shortname"].strip(), row["version"].strip())
        rec, how = resolution[key]
        row["match"] = match_value(rec, row)
        all_rows.append({**row, "_provider": f["provider"], "_file": f["path"].name, "_resolved": how, "_rec": rec})

    if not DRY_RUN:
        with f["path"].open("w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=f["fieldnames"] + ["match"], lineterminator="\n")
            w.writeheader()
            w.writerows(f["rows"])

print(f"{'Would write' if DRY_RUN else 'Wrote'} {len(files)} files, {len(all_rows)} rows")

Wrote 51 files, 3039 rows


## Step 7: Analysis

### 7a. Results by provider

In [8]:
def bucket(m):
    return m if m in ("true", "false") else "type_differs"


print(f"{'Provider':<11} {'rows':>5} {'true':>5} {'type≠':>6} {'false':>6} {'% true':>7} {'% in CMR assoc':>15}")
for prov in PROVIDERS + ["ALL"]:
    rs = [r for r in all_rows if prov in ("ALL", r["_provider"])]
    c = Counter(bucket(r["match"]) for r in rs)
    n = len(rs) or 1
    print(f"{prov:<11} {len(rs):>5} {c['true']:>5} {c['type_differs']:>6} {c['false']:>6} "
          f"{100 * c['true'] / n:>6.1f}% {100 * (c['true'] + c['type_differs']) / n:>14.1f}%")

Provider     rows  true  type≠  false  % true  % in CMR assoc
ASF           267    78     48    141   29.2%           47.2%
GES_DISC      812     0     10    802    0.0%            1.2%
GHRC_DAAC       2     0      0      2    0.0%            0.0%
LANCEMODIS    104     0      0    104    0.0%            0.0%
LARC_CLOUD   1836   121    117   1598    6.6%           13.0%
LPCLOUD        14     0      3     11    0.0%           21.4%
LPDAAC_ECS      4     0      0      4    0.0%            0.0%
ALL          3039   199    178   2662    6.5%           12.4%


### 7b. Results by file

In [9]:
print(f"{'File':<34} {'rows':>5} {'true':>5} {'type≠':>6} {'false':>6}")
for f in files:
    c = Counter(bucket(r["match"]) for r in f["rows"])
    print(f"{f['provider'] + '/' + f['path'].name:<34} {len(f['rows']):>5} {c['true']:>5} {c['type_differs']:>6} {c['false']:>6}")

File                                rows  true  type≠  false
ASF/ARIA_lineage.csv                   6     0      1      5
ASF/OPERA_lineage.csv                 40     6     17     17
ASF/SENTINEL-1C_lineage.csv          100    28     13     59
ASF/SENTINEL-1D_lineage.csv           98    42     14     42
ASF/SENTINEL-1_lineage.csv             8     0      3      5
ASF/UAVSAR_lineage.csv                15     2      0     13
GES_DISC/AIRS_lineage.csv            196     0      2    194
GES_DISC/CAR_lineage.csv              21     0      0     21
GES_DISC/LHSAS_lineage.csv             5     0      0      5
GES_DISC/MEaSUREs_OMI_lineage.csv     18     0      0     18
GES_DISC/MINDS_lineage.csv             5     0      0      5
GES_DISC/SNDR_lineage.csv            476     0      7    469
GES_DISC/TOMS_lineage.csv              6     0      0      6
GES_DISC/TOVS_lineage.csv             75     0      1     74
GES_DISC/TROPOMI_lineage.csv          10     0      0     10
GHRC_DAAC/glmmth_lineage

### 7c. Results by CSV `association_type`

In [10]:
print(f"{'CSV type':<10} {'rows':>5} {'true':>5} {'type≠':>6} {'false':>6}")
for t, n in Counter(r["association_type"] for r in all_rows).most_common():
    c = Counter(bucket(r["match"]) for r in all_rows if r["association_type"] == t)
    print(f"{t:<10} {n:>5} {c['true']:>5} {c['type_differs']:>6} {c['false']:>6}")

CSV type    rows  true  type≠  false
RELATED     1336    82     42   1212
PARENT       615     2     90    523
CHILD        584     0     46    538
INPUT        504   115      0    389


### 7d. Type differences: CSV `association_type` vs. CMR type

In [11]:
diff = Counter((r["association_type"], r["match"]) for r in all_rows if bucket(r["match"]) == "type_differs")
print(f"{'CSV type':<10} {'CMR type':<24} {'count':>5}")
for (csv_t, db_t), n in diff.most_common():
    print(f"{csv_t:<10} {db_t:<24} {n:>5}")

CSV type   CMR type                 count
PARENT     INPUT                       71
CHILD      DEPENDENT                   33
RELATED    PARENT                      19
PARENT     DEPENDENT                   14
CHILD      SCIENCE ASSOCIATED          13
RELATED    SCIENCE ASSOCIATED          12
RELATED    CHILD                       11
PARENT     SCIENCE ASSOCIATED           4
PARENT     RELATED                      1


### 7e. Why rows are `false`

A `false` row means either the source collection isn't in CMR, or its record has no association naming the related
collection. For the second case it helps to know whether the **related** collection exists in CMR at all:
- If it exists, the CSV link is plausible but CMR doesn't record it.
- If it doesn't exist, the CSV name may be wrong (a typo, or a guessed product name).

So the related short names are looked up too. This uses the same batched `shortNames` query, but across all
providers, because a related collection can belong to a different provider.

In [12]:
related_names = sorted({r["related_shortname"].strip() for r in all_rows})
t0 = time.time()
related_items = []
for i in range(0, len(related_names), BATCH_SIZE):
    related_items.extend(fetch_all({"shortNames": related_names[i:i + BATCH_SIZE]}))
related_in_cmr = defaultdict(set)   # short name -> providers that have it
for c in related_items:
    related_in_cmr[norm(c["shortName"])].add(c["provider"])
print(f"Looked up {len(related_names)} related short names in "
      f"{-(-len(related_names) // BATCH_SIZE)} requests ({time.time() - t0:.1f} s): "
      f"{sum(norm(n) in related_in_cmr for n in related_names)} exist in CMR\n")


def false_reason(r):
    if r["_rec"] is None:
        return "source collection not in CMR"
    if not r["_rec"].get("metadataAssociations"):
        return "source record has no metadataAssociations"
    if norm(r["related_shortname"]) in related_in_cmr:
        return "related collection exists in CMR, but not linked"
    return "related collection does not exist in CMR"


false_rows = [r for r in all_rows if r["match"] == "false"]
reasons = sorted({false_reason(r) for r in false_rows})
print(f"{'Provider':<11} {'false':>6}  " + "  ".join(f"{x[:28]:>28}" for x in reasons))
for prov in PROVIDERS + ["ALL"]:
    rs = [r for r in false_rows if prov in ("ALL", r["_provider"])]
    c = Counter(false_reason(r) for r in rs)
    print(f"{prov:<11} {len(rs):>6}  " + "  ".join(f"{c[x]:>28}" for x in reasons))

missing_related = Counter((r["_provider"], r["related_shortname"].strip()) for r in false_rows
                          if false_reason(r) == "related collection does not exist in CMR")
print(f"\n{len(missing_related)} distinct related short names not found in CMR. Per provider: "
      f"{dict(Counter(p for p, _ in missing_related))}")

Looked up 827 related short names in 9 requests (10.4 s): 819 exist in CMR

Provider     false  related collection does not   related collection exists in  source collection not in CMR
ASF            141                             0                           141                             0
GES_DISC       802                             0                           802                             0
GHRC_DAAC        2                             0                             2                             0
LANCEMODIS     104                            18                            80                             6
LARC_CLOUD    1598                             4                          1594                             0
LPCLOUD         11                             0                            11                             0
LPDAAC_ECS       4                             0                             4                             0
ALL           2662                            22    

### 7f. Related short names that don't exist in CMR

These CSV `related_shortname` values weren't found in CMR under any provider. They're likely wrong or guessed product
names in the ATBD-derived CSVs.

In [13]:
print(f"{'Provider':<11} {'CSV related_shortname':<24} {'rows':>4}")
for (prov, name), n in sorted(missing_related.items()):
    print(f"{prov:<11} {name:<24} {n:>4}")

Provider    CSV related_shortname    rows
LANCEMODIS  VNP02DNB_NRT                4
LANCEMODIS  VNP02MOD_NRT                4
LANCEMODIS  VNP03DNB_NRT                2
LANCEMODIS  VNP03MOD_NRT                2
LANCEMODIS  VNP14IMGTDL_NRT             2
LANCEMODIS  VNP14IMG_NRT                2
LANCEMODIS  VNP46A2_NRT                 2
LARC_CLOUD  CER_ES9_Terra+Aqua          4


### 7g. CMR associations the CSVs don't list

The `match` column only covers rows that are in the CSVs. Going the other way, these are the CMR `metadataAssociations`
of the source records that no CSV row names. They are split into:
- **self-references** (7g-i): `entryId` is the collection's own short name, usually another version of it, and
- **links to a different collection** (7g-ii).

In [14]:
# Key by CMR record: two CSV versions that resolve to the same record (Step 4 fallback) count once.
csv_related, rec_by_id = defaultdict(set), {}
for r in all_rows:
    if r["_rec"]:
        csv_related[r["_rec"]["conceptId"]].add(norm(r["related_shortname"]))
        rec_by_id[r["_rec"]["conceptId"]] = r["_rec"]

db_only = []
for cid, related in csv_related.items():
    rec = rec_by_id[cid]
    for a in rec.get("metadataAssociations") or []:
        if norm(a.get("entryId")) not in related:
            db_only.append((rec["provider"], rec["shortName"], rec["version"],
                            a.get("type"), (a.get("entryId") or "").strip(), a.get("version")))

self_refs = sorted((d for d in db_only if norm(d[4]) == norm(d[1])), key=str)
other_refs = sorted((d for d in db_only if norm(d[4]) != norm(d[1])), key=str)
csv_self_rows = sum(norm(r["shortname"]) == norm(r["related_shortname"]) for r in all_rows)
print(f"{len(db_only)} CMR associations not in the CSVs: {len(self_refs)} self-references, "
      f"{len(other_refs)} to a different collection")
print(f"(The CSVs themselves contain {csv_self_rows} self-referencing rows.)\n")
print(f"{'Provider':<11} {'self':>5} {'other':>6}")
for prov in PROVIDERS:
    print(f"{prov:<11} {sum(d[0] == prov for d in self_refs):>5} {sum(d[0] == prov for d in other_refs):>6}")

print(f"\nBy CMR type: self {dict(Counter(d[3] for d in self_refs).most_common())}")
print(f"             other {dict(Counter(d[3] for d in other_refs).most_common())}")


def print_grouped(items, fmt):
    # One line per source collection, listing its associations with `fmt`
    groups = defaultdict(list)
    for prov, name, ver, typ, entry, entry_ver in items:
        groups[(prov, name, ver)].append(fmt(typ, entry, entry_ver))
    print(f"{'Provider':<11} {'Collection':<32} {'ver':<8} associations")
    for (prov, name, ver), assocs in sorted(groups.items()):
        print(f"{prov:<11} {name:<32} {ver:<8} {'; '.join(assocs)}")
    return groups

1307 CMR associations not in the CSVs: 468 self-references, 839 to a different collection
(The CSVs themselves contain 0 self-referencing rows.)

Provider     self  other
ASF             0      9
GES_DISC      146      7
GHRC_DAAC       1      0
LANCEMODIS      6     10
LARC_CLOUD    315    811
LPCLOUD         0      0
LPDAAC_ECS      0      2

By CMR type: self {'PARENT': 241, 'SCIENCE ASSOCIATED': 149, 'CHILD': 42, 'INPUT': 24, None: 4, 'RELATED': 4, 'DEPENDENT': 4}
             other {'INPUT': 730, 'PARENT': 52, 'SCIENCE ASSOCIATED': 27, 'RELATED': 14, 'DEPENDENT': 11, 'CHILD': 5}


#### 7g-i. Associations that point back to the collection itself

The `entryId` is the record's own short name. Each one is classified by the association `version`:
- **other version:** points to a different version of the same collection, e.g. the CERES `PARENT` = "Preceding
  version" and `CHILD` = "Succeeding version" links. This is the normal, useful case.
- **same version:** points to the record itself. That's a self-loop that carries no lineage information.
- **no version:** the target can't be told apart from the record itself.

Versions are compared after normalizing case and leading zeros, so `02` = `2` and `V002` = `v2`. A link that only
differs in format is listed separately. For "other version" links, the target version is also looked up in CMR across
all providers (an older version can live under another provider). A missing version means the link points to a record
that public CMR doesn't have.

The CSVs contain no self-referencing rows. The ATBD-based lineage doesn't model version succession, so all of
these are absent from the CSVs by construction.

In [15]:
import re


def norm_ver(v):
    # "V02" -> "v2", "002" -> "2", "Edition4.2.1" -> "edition4.2.1"
    return re.sub(r"\d+", lambda m: str(int(m.group())), (v or "").strip().lower())


# Every version of each self-referenced short name, from any provider
self_names = sorted({d[1] for d in self_refs})
versions_in_cmr = defaultdict(set)   # short name -> raw versions present in CMR
for i in range(0, len(self_names), BATCH_SIZE):
    for c in fetch_all({"shortNames": self_names[i:i + BATCH_SIZE]}):
        versions_in_cmr[norm(c["shortName"])].add((c["version"] or "").strip())


def self_ref_kind(d):
    prov, name, ver, typ, entry, entry_ver = d
    if not (entry_ver or "").strip():
        return "no version"
    if norm_ver(entry_ver) == norm_ver(ver):
        return "same version"
    if norm_ver(entry_ver) in {norm_ver(v) for v in versions_in_cmr[norm(name)]}:
        return "other version"
    return "other version (not in CMR)"


def written_differently(d):
    # The target version exists, but not written the way CMR writes it (e.g. "02" for version "2")
    entry_ver = (d[5] or "").strip()
    known = versions_in_cmr[norm(d[1])]
    return bool(entry_ver) and entry_ver not in known and norm_ver(entry_ver) in {norm_ver(v) for v in known}


kinds = ["other version", "other version (not in CMR)", "same version", "no version"]
print(f"{'Provider':<11} " + " ".join(f"{k:>26}" for k in kinds))
for prov in PROVIDERS + ["ALL"]:
    c = Counter(self_ref_kind(d) for d in self_refs if prov in ("ALL", d[0]))
    print(f"{prov:<11} " + " ".join(f"{c[k]:>26}" for k in kinds))

print(f"\n{'CMR type':<20} " + " ".join(f"{k:>26}" for k in kinds))
for typ in sorted({str(d[3]) for d in self_refs}):
    c = Counter(self_ref_kind(d) for d in self_refs if str(d[3]) == typ)
    print(f"{typ:<20} " + " ".join(f"{c[k]:>26}" for k in kinds))

dangling = [d for d in self_refs if self_ref_kind(d) == "other version (not in CMR)"]
print(f"\n{len(dangling)} self-references point to a version that isn't in CMR:")
for prov, name, ver, typ, entry, entry_ver in dangling:
    print(f"  {prov:<11} {name:<32} v{ver:<12} {str(typ):<19} -> v{entry_ver:<12} (CMR has: "
          f"{', '.join(sorted(versions_in_cmr[norm(name)]))})")

fmt_diff = [d for d in self_refs if written_differently(d)]
print(f"\n{len(fmt_diff)} self-references write the version differently from CMR "
      f"(per provider: {dict(Counter(d[0] for d in fmt_diff))}), e.g.:")
for prov, name, ver, typ, entry, entry_ver in fmt_diff[:5]:
    print(f"  {prov:<11} {name:<32} v{ver:<8} {str(typ):<19} -> v{entry_ver}")

print(f"\nAll {len(self_refs)} self-references (TYPE -> target version):\n")
self_groups = print_grouped(self_refs, lambda typ, entry, v: f"{typ} -> {v or '(none)'}")
print(f"\n{len(self_groups)} collections have self-references")

Provider                 other version other version (not in CMR)               same version                 no version
ASF                                  0                          0                          0                          0
GES_DISC                             0                          0                        146                          0
GHRC_DAAC                            0                          1                          0                          0
LANCEMODIS                           0                          0                          0                          6
LARC_CLOUD                          93                        216                          6                          0
LPCLOUD                              0                          0                          0                          0
LPDAAC_ECS                           0                          0                          0                          0
ALL                                 93  

#### 7g-ii. Associations that point to a different collection

These are real lineage links in CMR that the ATBD-derived CSVs missed. They're candidates to review or add to the
lineage CSVs. Listed one line per source collection as `TYPE: entryId (version)`.

In [16]:
print("Most-linked CMR entryIds missing from the CSVs:")
for entry, n in Counter(d[4] for d in other_refs).most_common(10):
    print(f"  {n:>4}  {entry}")

print(f"\nAll {len(other_refs)} associations to a different collection:\n")
other_groups = print_grouped(other_refs, lambda typ, entry, v: f"{typ}: {entry}" + (f" ({v})" if v else ""))
print(f"\n{len(other_groups)} collections have links to other collections that the CSVs don't list")

Most-linked CMR entryIds missing from the CSVs:
   132  MICNFG
    54  MIANCAGP
    47  MIANCARP
    33  MIANLDBM
    32  MI1B2T
    31  MI1B2E
    31  MIB2GEOP
    30  MIANTASC
    28  MIRCCM
    24  MIANCSSC

All 839 associations to a different collection:

Provider    Collection                       ver      associations
ASF         OPERA_L2_CSLC-S1-STATIC_V1       1        INPUT: SENTINEL-1D_SLC
ASF         OPERA_L2_RTC-S1-STATIC_V1        1        INPUT: SENTINEL-1D_SLC
ASF         OPERA_L3_DIST-ALERT-S1_V1        1        SCIENCE ASSOCIATED: SENTINEL-1A_SLC (1); SCIENCE ASSOCIATED: SENTINEL-1C_SLC (1); SCIENCE ASSOCIATED: SENTINEL-1D_SLC (1)
ASF         OPERA_L4_TROPO-ZENITH_V1         1        INPUT: ASF_ECMWF_TROPO (1)
ASF         SENTINEL-1C_OCN                  1        SCIENCE ASSOCIATED: SENTINEL-1A_META_OCN
ASF         SENTINEL-1_BURSTS                1        INPUT: SENIINEL-1C_SLC; INPUT: SENIINEL-1D_SLC
GES_DISC    CAR_KUWAITOILFIRE_BRDF           2        SCIENCE ASSO

### 7h. Data quality of the CMR associations

These checks run on **every** `metadataAssociations` entry of the resolved source records, not only the ones
compared above. They find problems in CMR itself that make exact matching fail or make the links unusable:

| Check | Why it matters |
|---|---|
| Leading/trailing whitespace in `entryId` | Breaks exact matching (e.g. `"OPERA_L2_CSLC-S1-STATIC_V1 "`) |
| Free-text `entryId` (contains a space) | A description was put where a short name belongs |
| Missing `type` | The relationship is unknown |
| `type` not in the UMM-C enum | Invalid against the UMM-C schema |
| Duplicate entries | The same `entryId` + `version` + `type` is listed more than once in one record |
| Version inside `entryId` | e.g. `VNP46A1_1`, where the short name `VNP46A1` + version `1` was expected |
| `entryId` not in CMR | Typos, retired or non-public collections, e.g. `SENIINEL-1C_SLC`. Checked with a batched lookup across all providers. The CMR Search API (`collections.json?short_name=`) was spot-checked during development and agrees (e.g. `MICNFG`, `AIRMISR` return nothing) |

In [17]:
UMM_C_ASSOCIATION_TYPES = {"SCIENCE ASSOCIATED", "DEPENDENT", "INPUT", "PARENT", "CHILD", "RELATED",
                           "LARGER CITATION WORKS"}

assocs = [(rec["provider"], rec["shortName"], rec["version"], a)
          for rec in rec_by_id.values() for a in rec.get("metadataAssociations") or []]

# Look up every distinct entryId that isn't already known from Steps 3 and 7e
known_names = {norm(c["shortName"]) for c in cmr_items} | set(related_in_cmr)
to_lookup = sorted({(a.get("entryId") or "").strip() for *_, a in assocs} - {""})
to_lookup = [e for e in to_lookup if norm(e) not in known_names]
# Also look up the name before a trailing "_<suffix>" / ".<suffix>", for the version-inside-entryId check
to_lookup = sorted(set(to_lookup) | {e.rpartition(sep)[0] for e in to_lookup for sep in "_." if sep in e} - {""})
t0 = time.time()
for i in range(0, len(to_lookup), BATCH_SIZE):
    for c in fetch_all({"shortNames": to_lookup[i:i + BATCH_SIZE]}):
        known_names.add(norm(c["shortName"]))
print(f"{len(assocs)} association entries in {len(rec_by_id)} records; looked up {len(to_lookup)} extra names "
      f"in {-(-len(to_lookup) // BATCH_SIZE)} requests ({time.time() - t0:.1f} s)\n")


def embedded_version(entry, ver):
    # "VNP46A1_1" -> True when "VNP46A1" exists in CMR and the suffix looks like a version
    for sep in ("_", "."):
        base, _, suffix = entry.rpartition(sep)
        if base and norm(base) in known_names and (suffix == (ver or "").strip() or suffix.replace(".", "").isdigit()):
            return True
    return False


issues = defaultdict(list)
seen = Counter()
for prov, name, ver, a in assocs:
    raw, typ = a.get("entryId") or "", (a.get("type") or "").strip()
    entry = raw.strip()
    item = (prov, name, ver, a.get("type"), raw, a.get("version"))
    seen[(prov, name, ver, norm(entry), (a.get("version") or "").strip(), norm(typ))] += 1
    if raw != entry:
        issues["Leading/trailing whitespace in entryId"].append(item)
    if " " in entry:
        issues["Free-text entryId"].append(item)
    if not typ:
        issues["Missing type"].append(item)
    elif typ.upper() not in UMM_C_ASSOCIATION_TYPES:
        issues["type not in the UMM-C enum"].append(item)
    if norm(entry) not in known_names:
        if embedded_version(entry, a.get("version")):
            issues["Version inside entryId"].append(item)
        elif " " not in entry:
            issues["entryId not in CMR"].append(item)
for (prov, name, ver, entry, entry_ver, typ), n in seen.items():
    if n > 1:
        issues["Duplicate entries"].append((prov, name, ver, typ, f"{entry} (x{n})", entry_ver))

checks = ["Leading/trailing whitespace in entryId", "Free-text entryId", "Missing type", "type not in the UMM-C enum",
          "Duplicate entries", "Version inside entryId", "entryId not in CMR"]
print(f"{'Check':<40} {'entries':>7}  per provider")
for chk in checks:
    print(f"{chk:<40} {len(issues[chk]):>7}  {dict(Counter(i[0] for i in issues[chk]))}")

for chk in checks:
    if issues[chk]:
        print(f"\n{chk}:")
        for prov, name, ver, typ, entry, entry_ver in sorted(issues[chk], key=str):
            print(f"  {prov:<11} {name:<32} v{ver:<8} {str(typ):<19} {entry!r} ({entry_ver or '-'})")

missing_targets = Counter((i[0], i[4].strip()) for i in issues["entryId not in CMR"])
print(f"\n'entryId not in CMR' covers {len(missing_targets)} distinct (provider, entryId) targets. Most linked:")
for (prov, entry), n in missing_targets.most_common(15):
    print(f"  {n:>4}  {prov:<11} {entry}")

1685 association entries in 730 records; looked up 149 extra names in 2 requests (0.9 s)

Check                                    entries  per provider
Leading/trailing whitespace in entryId         5  {'ASF': 1, 'LANCEMODIS': 1, 'LARC_CLOUD': 3}
Free-text entryId                              1  {'LARC_CLOUD': 1}
Missing type                                   4  {'GES_DISC': 3, 'GHRC_DAAC': 1}
type not in the UMM-C enum                     0  {}
Duplicate entries                              0  {}
Version inside entryId                         8  {'LANCEMODIS': 5, 'LARC_CLOUD': 3}
entryId not in CMR                           445  {'ASF': 3, 'GES_DISC': 1, 'LARC_CLOUD': 441}

Leading/trailing whitespace in entryId:
  ASF         OPERA_L2_CSLC-S1_V1              v1        SCIENCE ASSOCIATED  'OPERA_L2_CSLC-S1-STATIC_V1 ' (1)
  LANCEMODIS  MOD09A1N                         v6.1NRT   SCIENCE ASSOCIATED  'MOD09A1N ' (-)
  LARC_CLOUD  MIB2GEOP                         v002      RELATED       

### 7i. Sample output: `lineage/metrics/LPCLOUD/AST_lineage.csv`

In [18]:
print((METRICS_DIR / "LPCLOUD" / "AST_lineage.csv").read_text())

shortname,version,association_type,related_shortname,related_version,confidence,lineage_relationship,match
AST_L1A,004,CHILD,AST14DEM,004,MEDIUM,downstream_product,false
AST_L1A,004,CHILD,AST_L1B,004,HIGH,downstream_product,DEPENDENT
AST_L1A,004,CHILD,AST_L1T,004,HIGH,downstream_product,false
AST_L1A,004,RELATED,AST_L1AE,003,MEDIUM,related_to,false
AST_L1B,004,PARENT,AST_L1A,004,HIGH,derived_from,INPUT
AST_L1B,004,CHILD,AST_07,004,LOW,downstream_product,false
AST_L1B,004,CHILD,AST_07XT,004,MEDIUM,downstream_product,false
AST_L1B,004,CHILD,AST_09,004,MEDIUM,downstream_product,false
AST_L1B,004,CHILD,AST_09T,004,LOW,downstream_product,false
AST_L1B,004,CHILD,AST_09XT,004,MEDIUM,downstream_product,false
AST_L1B,004,RELATED,AST_L1BE,003,MEDIUM,related_to,false
AST_L1B,004,RELATED,AST_L1T,004,MEDIUM,related_to,false
AST_L1T,004,PARENT,AST_L1A,004,HIGH,derived_from,INPUT
AST_L1T,004,RELATED,AST_L1B,004,MEDIUM,related_to,false



## Step 8: Summary

**What was done**
1. Read 51 `*_lineage.csv` files from the 7 provider directories under `lineage/metrics/`. That's 3,039 rows covering
   734 source collections. Any existing `match`/`Match` column was dropped on read and never used.
2. Queried CMR GraphQL directly with **batched requests** (provider + a `shortNames` list of up to 100). It took
   12 requests and about 13 s, and returned 787 records, all in the EOSDIS consortium.
   - One request per collection would have been 734 requests. Measured latency in test runs was 0.25–0.70 s per
     request, so about 3–9 minutes in sequence, with more exposure to throttling and transient errors.
   - A whole-provider fetch would have downloaded 5,610 records to use 734.
3. Resolved the source collections:
   - 730 matched exactly.
   - 2 matched through the single-version fallback. `VNP46A1_NRT` and `VNP46A1G_NRT` have CSV version 2, but CMR only has version 1.
   - 2 are not in CMR at all: `VNP46A2_NRT` and `VNP14IMGTDL_NRT`.
4. Computed `match` for every row and rewrote all 51 CSVs in place, with `match` as the last column.

**Results** (3,039 rows)

| Provider | rows | `true` | type differs | `false` | % `true` | % found in CMR |
|---|---:|---:|---:|---:|---:|---:|
| ASF | 267 | 78 | 48 | 141 | 29.2% | 47.2% |
| GES_DISC | 812 | 0 | 10 | 802 | 0.0% | 1.2% |
| GHRC_DAAC | 2 | 0 | 0 | 2 | 0.0% | 0.0% |
| LANCEMODIS | 104 | 0 | 0 | 104 | 0.0% | 0.0% |
| LARC_CLOUD | 1,836 | 121 | 117 | 1,598 | 6.6% | 13.0% |
| LPCLOUD | 14 | 0 | 3 | 11 | 0.0% | 21.4% |
| LPDAAC_ECS | 4 | 0 | 0 | 4 | 0.0% | 0.0% |
| **All** | **3,039** | **199** | **178** | **2,662** | **6.5%** | **12.4%** |

**Observations**
- **Most `false` rows are missing links, not bad names.** Of the 2,662 `false` rows, 2,634 point to a related
  collection that does exist in CMR, but the source record doesn't link to it. Only 22 rows name a collection that
  isn't in CMR. That's 8 names (7.f): 7 LANCEMODIS `VNP*_NRT` names and `CER_ES9_Terra+Aqua`. Another 6 rows come
  from the 2 source collections that aren't in CMR.
- **The type disagreements are a naming convention, not random.**
  - The CSVs use `PARENT`/`CHILD` for processing lineage, where CMR uses `INPUT`/`DEPENDENT`: PARENT→INPUT 71,
    CHILD→DEPENDENT 33, PARENT→DEPENDENT 14. That's 118 of the 178 type-differs rows.
  - In CMR, `PARENT`/`CHILD` mostly link versions of the same collection (241 self-referencing `PARENT` entries).
  - CSV rows typed `INPUT` agree best (115 `true`). CSV rows typed `CHILD` are never `true`.
  - Agreeing on one convention would turn most type-differs rows into `true`.
- **GES_DISC (0% `true`):** CMR associations there are mostly `SCIENCE ASSOCIATED` links to the record's own short
  name or its `_NRT` twin (146 self-references). They rarely link to the upstream products the CSVs list.
- **LANCEMODIS (0% `true`):** CMR links NRT products to standard products, often with the version inside the entry ID
  (`VNP46A1_1`, `VJ146A2_2`, `VJ114`). The CSVs link NRT products to other NRT products, so nothing lines up.
- **Reverse direction (7g):** 1,307 CMR associations aren't in the CSVs: 468 self-references and 839 links to a
  different collection.
- **Self-references (7g-i):** 468 entries in 449 collections. The CSVs contain none, because ATBD-based lineage
  doesn't model version succession.
  - 93 link to another version that exists in CMR. These are the useful version-succession links, mostly CERES/MISR `PARENT`.
  - 217 link to a version that public CMR doesn't have, e.g. `CER_BDS_*` Edition4 → Edition3, or the `g3b*` SAGE III
    chain (6 → 53 → 52). These are probably retired versions, so the links can't be followed.
  - 152 are self-loops (same version). 146 of them are GES_DISC `SCIENCE ASSOCIATED` placeholders that point
    to the record itself, which is why GES_DISC has 0% `true`.
  - 6 have no version (LANCEMODIS).
  - 20 write the version differently from CMR, e.g. `02` for `2`, or `EDITION2A` for `Edition2A`.
- **Links to other collections (7g-ii):** 839 entries in 228 collections. 811 are in LARC_CLOUD, mostly MISR `INPUT`
  lists (730 `INPUT` overall). These are candidates to add to the lineage CSVs. The most-linked one is `MICNFG`, from 132 collections.
- **CMR metadata data quality (7h)**, across all 1,685 association entries of the 730 resolved records:
  - **Link targets not in CMR:** the largest problem. 445 entries point to an `entryId` that public CMR
    doesn't have, 103 distinct targets. Most are MISR ancillary products (`MICNFG`, `MIANLDBM`, `MIANCSSC`, ...). Two are typos:
    - `SENIINEL-1C_SLC` and `SENIINEL-1D_SLC` in `SENTINEL-1_BURSTS`;
    - `ASF_ECMWF_TROPO` in `OPERA_L4_TROPO-ZENITH_V1`, where CMR's collection is `ASF_ECMWF_TROP`. The CSV has it right.
    - `PREFIRE_SAT1-2B-SFC` / `PREFIRE_SAT2-2B-SFC` (hyphen instead of underscore) in the PREFIRE L3 records (found by the Step 9 typo check).
  - **Version inside the entry ID** (8 entries), e.g. `VNP46A1_1`, `VJ146A2_2`, `DSCOVR_EPIC_L2_COMPOSITE_01`. `KUWAITOILFIRE_BRDF.2`
    also looks like this, but its base name isn't in CMR either.
  - **Leading/trailing whitespace** in 5 entry IDs, e.g. `"OPERA_L2_CSLC-S1-STATIC_V1 "` and `" TEMPO_CLDO4_L3_NRT"`.
    Matching here trims it.
  - **1 free-text entry ID:** `OPeNDAP data access for CAL_IIR_L2_Track-Standard-V4-20`.
  - **4 associations with no `type`:** LHASA and `glmmth`.
  - **No invalid types and no duplicate entries.**
- `LARC_CLOUD/SCAR_A_lineage.csv` has only a header row. It now has the `match` column but no data.

**How to re-run:** run all cells.
- Set `DRY_RUN = True` to compute without rewriting the CSVs.
- Set `RUN_BENCHMARK = False` to skip the Step 2 timing.
- Set `WRITE_REPORT = False` (Step 9) to skip writing the report.

Results reflect CMR at the retrieval time printed in Step 3.

## Step 9: Write the report

This step writes `lineage/reports/Collection_MetadataAssociations_CrossCheck_Report.md` from the variables computed
above, so every table and count in the report matches this run.
- **Computed** (refreshed on every run): the tables, counts, percentages, examples and DAAC error list. Likely typos
  are found by comparing each `entryId` that isn't in CMR with the known CMR short names (`difflib`, similarity ≥ 0.9,
  names of 10+ characters only, because short MISR codes such as `MI1B2EP` vs `MI1B2E` are real, distinct products).
  A match that differs only in digits (e.g. `CAL_LID_L1-Standard-V4-10` vs `-V4-51`) is a retired version, not a typo,
  so it isn't reported as one.
- **Fixed text:** the method description, the provider notes and the recommendations. These are the analyst's reading
  of the results, with the current numbers filled in. If the numbers shift a lot on a later run, re-read them before
  sharing the report.

Set `WRITE_REPORT = False` to skip writing the file.

In [19]:
import difflib

WRITE_REPORT = True
REPORT_PATH = REPO_ROOT / "lineage" / "reports" / "Collection_MetadataAssociations_CrossCheck_Report.md"
NOTEBOOK_REL = "../notebooks/lineage_association_cross_check.ipynb"


def n(x):
    return f"{x:,}"


def pct(a, b):
    return f"{100 * a / b:.1f}%" if b else "0.0%"


def md_table(headers, rows, align):
    # align: one character per column, "l" or "r"
    sep = ["---:" if a == "r" else "---" for a in align]
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(sep) + "|"]
    lines += ["| " + " | ".join(str(c) for c in row) + " |" for row in rows]
    return "\n".join(lines)


def counts(rows):
    c = Counter(bucket(r["match"]) for r in rows)
    return len(rows), c["true"], c["type_differs"], c["false"]


def code_list(items):
    return ", ".join(f"`{i}`" for i in items)


# ---- Numbers used in several sections ---------------------------------------
N, T, D, F = counts(all_rows)
n_sources = sum(map(len, sources.values()))
fr = Counter(false_reason(r) for r in false_rows)
type_diff = Counter((r["association_type"], r["match"]) for r in all_rows if bucket(r["match"]) == "type_differs")
convention = sum(type_diff[k] for k in [("PARENT", "INPUT"), ("CHILD", "DEPENDENT"), ("PARENT", "DEPENDENT")])
by_kind = Counter(self_ref_kind(d) for d in self_refs)
gesdisc_loops = sum(d[0] == "GES_DISC" and self_ref_kind(d) == "same version" for d in self_refs)
other_by_type = Counter(d[3] for d in other_refs)
larc_other = sum(d[0] == "LARC_CLOUD" for d in other_refs)
other_unresolvable = sum(norm(d[4]) not in known_names for d in other_refs)
related_requests = -(-len(related_names) // BATCH_SIZE)
date, hhmm = retrieved_at[:10], retrieved_at[11:16]

# Likely typos among the entryIds that aren't in CMR
display = {norm(c["shortName"]): c["shortName"] for c in [*cmr_items, *related_items]}
display.update({norm(r["related_shortname"]): r["related_shortname"].strip() for r in all_rows})
typos = []   # (provider, source short name, entryId, suggested short name, CSV has the suggestion)
for prov, name, ver, typ, entry, entry_ver in sorted(issues["entryId not in CMR"], key=str):
    e = entry.strip()
    if len(e) < 10:
        continue
    close = difflib.get_close_matches(norm(e), list(known_names), n=1, cutoff=0.9)
    if close and re.sub(r"[\d.]", "", norm(e)) != re.sub(r"[\d.]", "", close[0]):   # skip digit-only (version) differences
        csv_has = any(r["_provider"] == prov and r["shortname"] == name and norm(r["related_shortname"]) == close[0]
                      for r in all_rows)
        typos.append((prov, name, e, display.get(close[0], close[0]), csv_has))


def examples(check, k=4):
    c = Counter(i[4] for i in issues[check])
    return ", ".join(f'`"{e}"`' + (f" (×{m})" if m > 1 else "") if e != e.strip() or " " in e.strip()
                     else f"`{e}`" + (f" (×{m})" if m > 1 else "") for e, m in list(c.items())[:k])


seen_src = set()   # used to pick one typo example per source collection
sections = []
add = sections.append
BR = "  "   # markdown line break at the end of a header line

# ---- Header -------------------------------------------------------------------
add(f"""# Collection Lineage vs. CMR `metadataAssociations` Cross-Check Report

**Analysis Date:** {date}{BR}
**Data Source:** CMR GraphQL API (`{GRAPHQL_URL}`), queried live. Retrieved at {date} {hhmm} UTC{BR}
**Notebook:** [`lineage/notebooks/lineage_association_cross_check.ipynb`]({NOTEBOOK_REL}){BR}
**Scope:** `lineage/metrics/{{{",".join(PROVIDERS)}}}/*_lineage.csv`{BR}

---""")

# ---- 1. Executive summary -----------------------------------------------------------
add(f"""## 1. Executive Summary

The ATBD-derived lineage CSVs were checked against the `metadataAssociations` that CMR holds for each source
collection. That covers {len(files)} files, {n(N)} associations and {n(n_sources)} source collections. The results went
into a new `match` column in every CSV.

{md_table(["Result", "Rows", "Share"], [
    ["`true` (same related collection, same type)", n(T), pct(T, N)],
    ["Same related collection, **different type** (CMR type recorded)", n(D), pct(D, N)],
    ["`false` (related collection not in the CMR record)", n(F), pct(F, N)],
    ["**Total**", f"**{n(N)}**", ""]], "lrr")}

**Key findings**
1. **The CSVs and CMR mostly describe different links.** Only {pct(T + D, N)} of CSV associations appear in CMR at all.
   Almost every `false` row ({n(fr["related collection exists in CMR, but not linked"])} of {n(F)}) names a related
   collection that **does exist in CMR**, but the source record doesn't link to it. CMR's association metadata is
   sparse, and the CSV names are generally valid.
2. **Type disagreements are a convention mismatch.** The CSVs use `PARENT`/`CHILD` for processing lineage, while
   CMR providers use `INPUT`/`DEPENDENT` ({n(convention)} of {n(D)} type differences). In CMR, `PARENT`/`CHILD` mostly
   link versions of the same collection.
3. **CMR has {n(len(db_only))} associations the CSVs don't list.**
   - {n(len(self_refs))} are self-references, mostly version-succession links and GES_DISC self-loop placeholders.
   - {n(len(other_refs))} link to other collections, {n(larc_other)} of them in LARC_CLOUD (mostly MISR `INPUT` lists).
     These are candidates to add to the lineage CSVs.
4. **CMR association metadata has quality problems.**
   - {n(len(issues["entryId not in CMR"]))} association entries point to {n(len(missing_targets))} short names that
     public CMR doesn't have.
   - There are likely typos, such as {code_list(list(dict.fromkeys(t[2] for t in typos if (t[0], t[1]) not in seen_src and not seen_src.add((t[0], t[1]))))[:3])}.
   - Some entries put the version inside the entry ID, have stray whitespace, are free text, or have no type.

---""")

# ---- 2. Inputs --------------------------------------------------------------------------
input_rows = []
for prov in PROVIDERS:
    fs = [f for f in files if f["provider"] == prov]
    input_rows.append([prov, len(fs), n(sum(len(f["rows"]) for f in fs)), n(len(sources[prov]))])
input_rows.append(["**All**", f"**{len(files)}**", f"**{n(N)}**", f"**{n(n_sources)}**"])
header_only = [str(f["path"].relative_to(METRICS_DIR)) for f in files if not f["rows"]]
all_eosdis = all("EOSDIS" in (c["consortiums"] or []) for c in cmr_items)
add(f"""## 2. Inputs

{md_table(["Provider", "CSV files", "Rows", "Source collections"], input_rows, "lrrr")}

- Each directory name is a CMR provider ID. {"All" if all_eosdis else "Not all"} {n(len(cmr_items))} CMR records
  retrieved belong to the EOSDIS consortium.
- Any existing `match`/`Match` column was **dropped on read and never used**. The new column was computed only from CMR.
""" + "".join(f"- `{h}` has only a header row. It got the `match` column but has no data.\n" for h in header_only)
    + "\n---")

# ---- 3. Method ----------------------------------------------------------------------------
if RUN_BENCHMARK:
    est = n_sources * mean
    option_a = f"Mean {mean:.2f} s per request (max {max(latencies):.2f} s) in this run, so about {est / 60:.1f} min sequentially"
    whole = sum(totals.values())
    worst = max(PROVIDERS, key=lambda p: totals[p] / max(len(sources[p]), 1))
    option_b = (f"| B. Whole provider (paged) | {len(PROVIDERS)}+ | {n(whole)} | Downloads {whole / n_sources:.1f}× more records "
                f"than needed (e.g. {worst}: {n(totals[worst])} records for {len(sources[worst])} "
                f"collection{'s' if len(sources[worst]) != 1 else ''}) |")
else:
    option_a = "Not measured in this run (`RUN_BENCHMARK = False`)"
    option_b = f"| B. Whole provider (paged) | {len(PROVIDERS)}+ | all records of {len(PROVIDERS)} providers | Not measured in this run |"

res = Counter(how for _, how in resolution.values())
fallbacks = sorted((k, rec) for k, (rec, how) in resolution.items() if how == "single_version_fallback")
not_found = sorted(k for k, (rec, how) in resolution.items() if how in ("not_in_cmr", "version_ambiguous"))
res_rows = [["Exact match (provider + short name + version)", n(res["exact"])]]
res_rows.append(["Single-version fallback: CSV version not in CMR, but only one version exists, so it was used",
                 f"{len(fallbacks)}" + (" (" + "; ".join(f"`{k[1]}`: CSV v{k[2]}, CMR v{rec['version']}" for k, rec in fallbacks) + ")"
                                        if fallbacks else "")])
res_rows.append(["Not in CMR (all rows `false`)",
                 f"{len(not_found)}" + (" (" + code_list(k[1] for k in not_found) + ")" if not_found else "")])

add(f"""## 3. Method

### 3.1 Query strategy

Three options were evaluated for getting `metadataAssociations` for {n(n_sources)} collections:

| Strategy | Requests | Records downloaded | Measured / estimated time |
|---|---:|---:|---|
| A. One request per collection | {n(n_sources)} | {n(n_sources)} | {option_a} |
{option_b}
| **C. Batched: provider + `shortNames` list (chosen)** | **{n_requests}** | **{n(len(cmr_items))}** | **{fetch_seconds:.1f} s in this run** |

Why option C:
- **Fewest requests and least wasted data.** `CollectionsInput.shortNames` accepts a list, so one request covers up to
  {BATCH_SIZE} short names.
- **Querying by short name, not entry ID, returns every version.** That makes it possible to recover from a wrong
  version in a CSV (section 3.2).
- **Robust.** It retries with exponential backoff, and it pages with `cursor` until `count` items are received.
- **Lighter on CMR.** It sends {n_requests} requests to a shared public service instead of {n(n_sources)}.

The same batched pattern was reused for the extra analysis lookups: {n(len(related_names))} related short names in
{related_requests} requests, all self-referenced versions, and {n(len(to_lookup))} association targets.

### 3.2 Resolving source collections

{md_table(["Resolution", "Collections"], res_rows, "lr")}

### 3.3 Matching rules

1. Find the source record's `metadataAssociations` entries whose `entryId` equals the CSV `related_shortname`. The
   comparison ignores case and surrounding whitespace.
2. If several entries share that `entryId`, keep the one whose `version` equals the CSV `related_version`. CMR often
   links several versions of one short name with different types.
3. If the type is the same, the value is `true`. If the type differs, the value is the CMR type (several types are
   joined with `|`). If there's no entry, the value is `false`.

---""")

# ---- 4. Results ---------------------------------------------------------------------------
prov_rows = []
for prov in PROVIDERS + ["ALL"]:
    rs = [r for r in all_rows if prov in ("ALL", r["_provider"])]
    rn, t, d, f = counts(rs)
    row = [prov, n(rn), n(t), n(d), n(f), pct(t, rn), pct(t + d, rn)]
    prov_rows.append(row if prov != "ALL" else ["**All**"] + [f"**{c}**" for c in row[1:]])

file_stats = [(f["provider"] + "/" + f["path"].name.removesuffix("_lineage.csv"), *counts(f["rows"])) for f in files]
top_files = sorted((s for s in file_stats if s[2] > 0), key=lambda s: (-s[2], -s[1]))[:6]
no_agree = sorted((s for s in file_stats if s[2] == 0 and s[1] > 0), key=lambda s: -s[1])[:3]

type_rows, type_true_share = [], {}
for t, cnt in Counter(r["association_type"] for r in all_rows).most_common():
    _, tt, td, tf = counts([r for r in all_rows if r["association_type"] == t])
    type_rows.append([t, n(cnt), n(tt), n(td), n(tf)])
    type_true_share[t] = tt / cnt
best_type = max(type_true_share, key=type_true_share.get)
never = [t for t, s in type_true_share.items() if s == 0]
top2 = type_diff.most_common(2)

reason_rows = [["Related collection exists in CMR, but the source record doesn't link to it",
                n(fr["related collection exists in CMR, but not linked"])],
               ["Related collection doesn't exist in CMR", n(fr["related collection does not exist in CMR"])],
               ["Source collection not in CMR", n(fr["source collection not in CMR"])]]
if fr["source record has no metadataAssociations"]:
    reason_rows.append(["Source record has no `metadataAssociations`", n(fr["source record has no metadataAssociations"])])

pct_true = {}
for p in PROVIDERS:
    pn, pt, _, _ = counts([r for r in all_rows if r["_provider"] == p])
    pct_true[p] = pct(pt, pn)

add(f"""## 4. Results

### 4.1 By provider

{md_table(["Provider", "Rows", "`true`", "Type differs", "`false`", "% `true`", "% found in CMR"], prov_rows, "lrrrrrr")}

### 4.2 Collection suites with the most agreement

{md_table(["File", "Rows", "`true`", "Type differs", "`false`"], [[s[0], n(s[1]), n(s[2]), n(s[3]), n(s[4])] for s in top_files], "lrrrr")}

Several large suites have **no** exact CMR agreement, including
{", ".join(f"{s[0]} ({n(s[1])} rows" + (f", of which {n(s[3])} differ by type)" if s[3] else ")") for s in no_agree)}.
The per-file table for all {len(files)} files is in notebook section 7b.

### 4.3 By CSV association type

{md_table(["CSV type", "Rows", "`true`", "Type differs", "`false`"], type_rows, "lrrrr")}

`{best_type}` is the CSV type that agrees with CMR best ({100 * type_true_share[best_type]:.1f}% `true`).""" +
    (f" {' and '.join(f'`{t}`' for t in never)} never match{'es' if len(never) == 1 else ''} exactly." if never else "") + f"""

### 4.4 Type differences (CSV type → CMR type)

{md_table(["CSV type", "CMR type", "Count"], [[a, b, n(c)] for (a, b), c in type_diff.most_common()], "llr")}

{" and ".join(f"{a}→{b}" for (a, b), _ in top2)} together account for {n(sum(c for _, c in top2))} of the {n(D)}
differences. Adopting CMR's convention in the lineage skill (upstream = `INPUT`, downstream = `DEPENDENT`) would turn
most of these rows into `true`.

### 4.5 Why rows are `false`

{md_table(["Reason", "Rows"], reason_rows, "lr")}

The {n(sum(missing_related.values()))} rows with a non-existent related collection use {len(missing_related)} short
names. These are probably guessed product names in the CSVs:

{md_table(["Provider", "CSV `related_shortname`", "Rows"], [[p, s, c] for (p, s), c in sorted(missing_related.items())], "llr")}

### 4.6 Provider notes

- **GES_DISC ({pct_true["GES_DISC"]} `true`):** CMR associations are mostly `SCIENCE ASSOCIATED` entries that point to the
  record itself or to its `_NRT` twin. There are {gesdisc_loops} self-loops. They rarely name the upstream products the CSVs list.
- **LANCEMODIS ({pct_true["LANCEMODIS"]} `true`):** CMR links NRT products to *standard* products, often with the version
  inside the entry ID (e.g. {code_list(sorted({i[4] for i in issues["Version inside entryId"] if i[0] == "LANCEMODIS"})[:3])}).
  The CSVs link NRT products to other NRT products.
- **ASF ({pct_true["ASF"]} `true`):** the best agreement, mainly from the SENTINEL-1C/1D suites.

---""")

# ---- 5. CMR associations not in the CSVs --------------------------------------------
split_rows = [[p, n(sum(d[0] == p for d in self_refs)), n(sum(d[0] == p for d in other_refs))] for p in PROVIDERS]
split_rows.append(["**All**", f"**{n(len(self_refs))}**", f"**{n(len(other_refs))}**"])


def kind_examples(kind, k=4):
    seen, out = set(), []
    for prov, name, ver, typ, entry, entry_ver in self_refs:
        family = name[:6]
        if self_ref_kind((prov, name, ver, typ, entry, entry_ver)) == kind and family not in seen:
            seen.add(family)
            out.append(f"`{name}` {ver} → {entry_ver}")
        if len(out) == k:
            break
    return ", ".join(out)


kind_rows = [
    ["Links to another version that exists in CMR", n(by_kind["other version"]),
     "Version-succession links, e.g. " + kind_examples("other version", 2)],
    ["Links to a version public CMR doesn't have", n(by_kind["other version (not in CMR)"]),
     "e.g. " + kind_examples("other version (not in CMR)") + ". Probably retired versions, so the links can't be followed"],
    ["Points to the record itself (same version)", n(by_kind["same version"]),
     f"{gesdisc_loops} are GES_DISC placeholders. They carry no lineage information"],
    ["No version given", n(by_kind["no version"]),
     f"{', '.join(sorted({d[0] for d in self_refs if self_ref_kind(d) == 'no version'})) or '-'}. "
     "The target can't be told apart from the record itself"],
]
top_linked = Counter(d[4] for d in other_refs).most_common(6)

add(f"""## 5. CMR Associations Not in the CSVs

The `match` column only shows CSV rows. Going the other way, CMR has **{n(len(db_only))}** associations on the source
records that no CSV row names.

{md_table(["Provider", "Self-references", "To another collection"], split_rows, "lrr")}

### 5.1 Self-referencing associations ({n(len(self_refs))} entries, {n(len(self_groups))} collections)

The `entryId` is the record's own short name. The CSVs contain **{"no" if csv_self_rows == 0 else n(csv_self_rows)}**
self-references, because ATBD-based lineage doesn't model version succession.

{md_table(["Kind", "Entries", "Notes"], kind_rows, "lrl")}

Another {len(fmt_diff)} self-references write the version differently from CMR, e.g.
{", ".join(f"`{d[5]}` for version `{d[2]}` ({d[1]})" for d in list({d[0]: d for d in reversed(fmt_diff)}.values())[:2])}. They resolve once versions are normalized,
but they would fail an exact-match lookup.

### 5.2 Associations to a different collection ({n(len(other_refs))} entries, {n(len(other_groups))} collections)

These are lineage links already in CMR that the ATBD-derived CSVs missed. **They're the main candidates to add to the
lineage CSVs.**

- By type: {", ".join(f"`{t}` {n(c)}" for t, c in other_by_type.most_common())}.
- {n(larc_other)} are in LARC_CLOUD. The most-linked targets are {", ".join(f"`{e}` ({c})" for e, c in top_linked)}.
- {n(other_unresolvable)} of the {n(len(other_refs))} have an `entryId` that doesn't resolve to a CMR collection: the
  target isn't in CMR, or the ID has a version inside it or is free text (section 6). Adding those to the CSVs as-is
  would produce links that can't be resolved.

---""")

# ---- 6. Data quality ----------------------------------------------------------------------
dq_rows = [
    ["`entryId` not in CMR", f"{n(len(issues['entryId not in CMR']))} ({n(len(missing_targets))} distinct targets)",
     ", ".join(f"`{e}` ({c})" for (_, e), c in missing_targets.most_common(3))
     + (", likely typos " + code_list(sorted({t[2] for t in typos})) if typos else "")],
    ["Version inside `entryId`", n(len(issues["Version inside entryId"])), examples("Version inside entryId", 5)],
    ["Leading/trailing whitespace in `entryId`", n(len(issues["Leading/trailing whitespace in entryId"])),
     examples("Leading/trailing whitespace in entryId", 5)],
    ["Missing `type`", n(len(issues["Missing type"])), examples("Missing type")],
    ["Free-text `entryId`", n(len(issues["Free-text entryId"])),
     ", ".join(f'`"{i[4].strip()}"` in `{i[1]}`' for i in issues["Free-text entryId"])],
    ["`type` not in the UMM-C enum", n(len(issues["type not in the UMM-C enum"])), examples("type not in the UMM-C enum")],
    ["Duplicate entries", n(len(issues["Duplicate entries"])), examples("Duplicate entries")],
]

daac_errors = []
typo_groups = defaultdict(list)
for prov, name, e, suggestion, csv_has in typos:
    typo_groups[(prov, name, csv_has)].append((e, suggestion))
for (prov, name, csv_has), pairs in sorted(typo_groups.items()):
    daac_errors.append(f"- **{prov} `{name}`:** " + "; ".join(f"`{e}` is not in CMR, probably `{s}`" for e, s in pairs)
                       + (". The CSV has the correct name." if csv_has else "."))
for prov, name, ver, typ, entry, entry_ver in issues["Free-text entryId"]:
    daac_errors.append(f"- **{prov} `{name}`:** the `{typ}` entry ID is a description, not a short name.")
vi = defaultdict(set)
for prov, name, ver, typ, entry, entry_ver in issues["Version inside entryId"]:
    vi[prov].add(entry.strip())
for prov, entries in sorted(vi.items()):
    daac_errors.append(f"- **{prov}:** entry IDs with the version inside them ({code_list(sorted(entries))}) should be "
                       "split into short name + `Version`.")

add(f"""## 6. CMR Association Data Quality

All {n(len(assocs))} association entries on the {n(len(rec_by_id))} resolved records were checked:

{md_table(["Check", "Entries", "Examples"], dq_rows, "lrl")}

Specific errors worth reporting to the DAACs:
{chr(10).join(daac_errors) if daac_errors else "- None found in this run."}

"Not in CMR" means that CMR GraphQL returned no collection with that short name under any provider. During development
a sample (e.g. `MICNFG`, `AIRMISR`, `ASF_ECMWF_TROPO`) was confirmed against the CMR Search API
(`collections.json?short_name=`), which returns nothing for them either.

---""")

# ---- 7. Recommendations & 8. Reproducing ------------------------------------------------
add(f"""## 7. Recommendations

**For the lineage CSV generation skill (`Collection_Lineage_via_ATBD_Skill`)**
1. Use CMR's type convention: upstream processing inputs = `INPUT`, downstream products = `DEPENDENT`. Keep
   `PARENT`/`CHILD` for version succession. This alone would resolve most of the {n(D)} type differences.
2. Check every `related_shortname` against CMR before writing it. That would drop or fix the {len(missing_related)}
   non-existent names.
3. Merge in the {n(len(other_refs) - other_unresolvable)} CMR links to other collections whose target exists in CMR
   (section 5.2). Leave out, or first correct, the {n(other_unresolvable)} whose `entryId` doesn't resolve.

**For the DAACs / CMR metadata**
1. Fix the likely typos and the free-text entry IDs listed in section 6.
2. Trim whitespace from entry IDs, and split versions out of entry IDs into the `Version` field.
3. Add `Type` to the {len(issues["Missing type"])} untyped associations.
4. Review the {n(by_kind["other version (not in CMR)"])} version links to versions that no longer exist, and the
   {gesdisc_loops} GES_DISC self-loop placeholders.
5. Review the {n(len(missing_targets))} association targets that public CMR doesn't have (most-linked:
   {code_list(e for (_, e), _ in missing_targets.most_common(3))}).

---

## 8. Reproducing the Results

Run every cell of `lineage/notebooks/lineage_association_cross_check.ipynb`. It uses the `uv_metadata` environment
(Python 3.14, `requests`) and needs no authentication. Step 9 of the notebook regenerates this report.
- `DRY_RUN = True` computes everything without rewriting the CSVs.
- `RUN_BENCHMARK = False` skips the query-strategy timing.
- `WRITE_REPORT = False` skips writing this report.

Results reflect CMR at the retrieval time shown at the top of this report.
""")

report = "\n\n".join(sections)
if WRITE_REPORT:
    REPORT_PATH.write_text(report, encoding="utf-8")
print(f"{'Wrote' if WRITE_REPORT else 'Would write'} {REPORT_PATH.relative_to(REPO_ROOT)} "
      f"({len(report.splitlines())} lines, {len(sections)} sections)")
print(f"Likely typos found: {[(t[2], t[3]) for t in typos]}")

Wrote lineage/reports/Collection_MetadataAssociations_CrossCheck_Report.md (301 lines, 8 sections)
Likely typos found: [('ASF_ECMWF_TROPO', 'ASF_ECMWF_TROP'), ('SENIINEL-1C_SLC', 'SENTINEL-1C_SLC'), ('SENIINEL-1D_SLC', 'SENTINEL-1D_SLC'), ('PREFIRE_SAT1-2B-SFC', 'PREFIRE_SAT1_2B-SFC'), ('PREFIRE_SAT2-2B-SFC', 'PREFIRE_SAT2_2B-SFC'), ('SAGE2_V6.20_AEROSOL_O3_NO2_H2O_BINARY', 'SAGE2_AEROSOL_O3_NO2_H2O_BINARY')]


In [20]:
# Preview the report as rendered markdown
from IPython.display import Markdown, display

display(Markdown(report))

# Collection Lineage vs. CMR `metadataAssociations` Cross-Check Report

**Analysis Date:** 2026-11-24  
**Data Source:** CMR GraphQL API (`https://graphql.earthdata.nasa.gov/api`), queried live. Retrieved at 2026-11-24 22:35 UTC  
**Notebook:** [`lineage/notebooks/lineage_association_cross_check.ipynb`](../notebooks/lineage_association_cross_check.ipynb)  
**Scope:** `lineage/metrics/{ASF,GES_DISC,GHRC_DAAC,LANCEMODIS,LARC_CLOUD,LPCLOUD,LPDAAC_ECS}/*_lineage.csv`  

---

## 1. Executive Summary

The ATBD-derived lineage CSVs were checked against the `metadataAssociations` that CMR holds for each source
collection. That covers 51 files, 3,039 associations and 734 source collections. The results went
into a new `match` column in every CSV.

| Result | Rows | Share |
|---|---:|---:|
| `true` (same related collection, same type) | 199 | 6.5% |
| Same related collection, **different type** (CMR type recorded) | 178 | 5.9% |
| `false` (related collection not in the CMR record) | 2,662 | 87.6% |
| **Total** | **3,039** |  |

**Key findings**
1. **The CSVs and CMR mostly describe different links.** Only 12.4% of CSV associations appear in CMR at all.
   Almost every `false` row (2,634 of 2,662) names a related
   collection that **does exist in CMR**, but the source record doesn't link to it. CMR's association metadata is
   sparse, and the CSV names are generally valid.
2. **Type disagreements are a convention mismatch.** The CSVs use `PARENT`/`CHILD` for processing lineage, while
   CMR providers use `INPUT`/`DEPENDENT` (118 of 178 type differences). In CMR, `PARENT`/`CHILD` mostly
   link versions of the same collection.
3. **CMR has 1,307 associations the CSVs don't list.**
   - 468 are self-references, mostly version-succession links and GES_DISC self-loop placeholders.
   - 839 link to other collections, 811 of them in LARC_CLOUD (mostly MISR `INPUT` lists).
     These are candidates to add to the lineage CSVs.
4. **CMR association metadata has quality problems.**
   - 445 association entries point to 103 short names that
     public CMR doesn't have.
   - There are likely typos, such as `ASF_ECMWF_TROPO`, `SENIINEL-1C_SLC`, `PREFIRE_SAT1-2B-SFC`.
   - Some entries put the version inside the entry ID, have stray whitespace, are free text, or have no type.

---

## 2. Inputs

| Provider | CSV files | Rows | Source collections |
|---|---:|---:|---:|
| ASF | 6 | 267 | 41 |
| GES_DISC | 9 | 812 | 157 |
| GHRC_DAAC | 1 | 2 | 1 |
| LANCEMODIS | 4 | 104 | 20 |
| LARC_CLOUD | 29 | 1,836 | 510 |
| LPCLOUD | 1 | 14 | 3 |
| LPDAAC_ECS | 1 | 4 | 2 |
| **All** | **51** | **3,039** | **734** |

- Each directory name is a CMR provider ID. All 787 CMR records
  retrieved belong to the EOSDIS consortium.
- Any existing `match`/`Match` column was **dropped on read and never used**. The new column was computed only from CMR.
- `LARC_CLOUD/SCAR_A_lineage.csv` has only a header row. It got the `match` column but has no data.

---

## 3. Method

### 3.1 Query strategy

Three options were evaluated for getting `metadataAssociations` for 734 collections:

| Strategy | Requests | Records downloaded | Measured / estimated time |
|---|---:|---:|---|
| A. One request per collection | 734 | 734 | Mean 0.45 s per request (max 0.73 s) in this run, so about 5.5 min sequentially |
| B. Whole provider (paged) | 7+ | 5,610 | Downloads 7.6× more records than needed (e.g. GHRC_DAAC: 703 records for 1 collection) |
| **C. Batched: provider + `shortNames` list (chosen)** | **12** | **787** | **11.2 s in this run** |

Why option C:
- **Fewest requests and least wasted data.** `CollectionsInput.shortNames` accepts a list, so one request covers up to
  100 short names.
- **Querying by short name, not entry ID, returns every version.** That makes it possible to recover from a wrong
  version in a CSV (section 3.2).
- **Robust.** It retries with exponential backoff, and it pages with `cursor` until `count` items are received.
- **Lighter on CMR.** It sends 12 requests to a shared public service instead of 734.

The same batched pattern was reused for the extra analysis lookups: 827 related short names in
9 requests, all self-referenced versions, and 149 association targets.

### 3.2 Resolving source collections

| Resolution | Collections |
|---|---:|
| Exact match (provider + short name + version) | 730 |
| Single-version fallback: CSV version not in CMR, but only one version exists, so it was used | 2 (`VNP46A1G_NRT`: CSV v2, CMR v1; `VNP46A1_NRT`: CSV v2, CMR v1) |
| Not in CMR (all rows `false`) | 2 (`VNP14IMGTDL_NRT`, `VNP46A2_NRT`) |

### 3.3 Matching rules

1. Find the source record's `metadataAssociations` entries whose `entryId` equals the CSV `related_shortname`. The
   comparison ignores case and surrounding whitespace.
2. If several entries share that `entryId`, keep the one whose `version` equals the CSV `related_version`. CMR often
   links several versions of one short name with different types.
3. If the type is the same, the value is `true`. If the type differs, the value is the CMR type (several types are
   joined with `|`). If there's no entry, the value is `false`.

---

## 4. Results

### 4.1 By provider

| Provider | Rows | `true` | Type differs | `false` | % `true` | % found in CMR |
|---|---:|---:|---:|---:|---:|---:|
| ASF | 267 | 78 | 48 | 141 | 29.2% | 47.2% |
| GES_DISC | 812 | 0 | 10 | 802 | 0.0% | 1.2% |
| GHRC_DAAC | 2 | 0 | 0 | 2 | 0.0% | 0.0% |
| LANCEMODIS | 104 | 0 | 0 | 104 | 0.0% | 0.0% |
| LARC_CLOUD | 1,836 | 121 | 117 | 1,598 | 6.6% | 13.0% |
| LPCLOUD | 14 | 0 | 3 | 11 | 0.0% | 21.4% |
| LPDAAC_ECS | 4 | 0 | 0 | 4 | 0.0% | 0.0% |
| **All** | **3,039** | **199** | **178** | **2,662** | **6.5%** | **12.4%** |

### 4.2 Collection suites with the most agreement

| File | Rows | `true` | Type differs | `false` |
|---|---:|---:|---:|---:|
| LARC_CLOUD/MISR | 383 | 88 | 37 | 258 |
| ASF/SENTINEL-1D | 98 | 42 | 14 | 42 |
| ASF/SENTINEL-1C | 100 | 28 | 13 | 59 |
| LARC_CLOUD/PREFIRE | 166 | 21 | 44 | 101 |
| LARC_CLOUD/TEMPO | 169 | 7 | 0 | 162 |
| ASF/OPERA | 40 | 6 | 17 | 17 |

Several large suites have **no** exact CMR agreement, including
GES_DISC/SNDR (476 rows, of which 7 differ by type), LARC_CLOUD/CERES (268 rows), LARC_CLOUD/CALIPSO (219 rows, of which 22 differ by type).
The per-file table for all 51 files is in notebook section 7b.

### 4.3 By CSV association type

| CSV type | Rows | `true` | Type differs | `false` |
|---|---:|---:|---:|---:|
| RELATED | 1,336 | 82 | 42 | 1,212 |
| PARENT | 615 | 2 | 90 | 523 |
| CHILD | 584 | 0 | 46 | 538 |
| INPUT | 504 | 115 | 0 | 389 |

`INPUT` is the CSV type that agrees with CMR best (22.8% `true`). `CHILD` never matches exactly.

### 4.4 Type differences (CSV type → CMR type)

| CSV type | CMR type | Count |
|---|---|---:|
| PARENT | INPUT | 71 |
| CHILD | DEPENDENT | 33 |
| RELATED | PARENT | 19 |
| PARENT | DEPENDENT | 14 |
| CHILD | SCIENCE ASSOCIATED | 13 |
| RELATED | SCIENCE ASSOCIATED | 12 |
| RELATED | CHILD | 11 |
| PARENT | SCIENCE ASSOCIATED | 4 |
| PARENT | RELATED | 1 |

PARENT→INPUT and CHILD→DEPENDENT together account for 104 of the 178
differences. Adopting CMR's convention in the lineage skill (upstream = `INPUT`, downstream = `DEPENDENT`) would turn
most of these rows into `true`.

### 4.5 Why rows are `false`

| Reason | Rows |
|---|---:|
| Related collection exists in CMR, but the source record doesn't link to it | 2,634 |
| Related collection doesn't exist in CMR | 22 |
| Source collection not in CMR | 6 |

The 22 rows with a non-existent related collection use 8 short
names. These are probably guessed product names in the CSVs:

| Provider | CSV `related_shortname` | Rows |
|---|---|---:|
| LANCEMODIS | VNP02DNB_NRT | 4 |
| LANCEMODIS | VNP02MOD_NRT | 4 |
| LANCEMODIS | VNP03DNB_NRT | 2 |
| LANCEMODIS | VNP03MOD_NRT | 2 |
| LANCEMODIS | VNP14IMGTDL_NRT | 2 |
| LANCEMODIS | VNP14IMG_NRT | 2 |
| LANCEMODIS | VNP46A2_NRT | 2 |
| LARC_CLOUD | CER_ES9_Terra+Aqua | 4 |

### 4.6 Provider notes

- **GES_DISC (0.0% `true`):** CMR associations are mostly `SCIENCE ASSOCIATED` entries that point to the
  record itself or to its `_NRT` twin. There are 146 self-loops. They rarely name the upstream products the CSVs list.
- **LANCEMODIS (0.0% `true`):** CMR links NRT products to *standard* products, often with the version
  inside the entry ID (e.g. `VJ146A2_2`, `VNP46A1_1`, `VNP46A1_2`).
  The CSVs link NRT products to other NRT products.
- **ASF (29.2% `true`):** the best agreement, mainly from the SENTINEL-1C/1D suites.

---

## 5. CMR Associations Not in the CSVs

The `match` column only shows CSV rows. Going the other way, CMR has **1,307** associations on the source
records that no CSV row names.

| Provider | Self-references | To another collection |
|---|---:|---:|
| ASF | 0 | 9 |
| GES_DISC | 146 | 7 |
| GHRC_DAAC | 1 | 0 |
| LANCEMODIS | 6 | 10 |
| LARC_CLOUD | 315 | 811 |
| LPCLOUD | 0 | 0 |
| LPDAAC_ECS | 0 | 2 |
| **All** | **468** | **839** |

### 5.1 Self-referencing associations (468 entries, 449 collections)

The `entryId` is the record's own short name. The CSVs contain **no**
self-references, because ATBD-based lineage doesn't model version succession.

| Kind | Entries | Notes |
|---|---:|---|
| Links to another version that exists in CMR | 93 | Version-succession links, e.g. `CER_BDS_NPP-FM5` Edition2 → Edition1-CV, `CER_CRS_Terra-FM1-MODIS` Edition4A → Edition2G |
| Links to a version public CMR doesn't have | 217 | e.g. `glmmth` 1 → 3, `AirMSPI_ORACLES_Ellipsoid-projected_Georegistered_Radiance_Data` 6 → 1, `CERES_EBAF` Edition4.2.1 → Edition4.2, `CER_BDS_Aqua-FM3` Edition4 → Edition3. Probably retired versions, so the links can't be followed |
| Points to the record itself (same version) | 152 | 146 are GES_DISC placeholders. They carry no lineage information |
| No version given | 6 | LANCEMODIS. The target can't be told apart from the record itself |

Another 20 self-references write the version differently from CMR, e.g.
`EDITION2A` for version `Edition2B` (CER_SSF_NPP-FM5-VIIRS), `02` for version `2` (SNDRJ1ATMSL1B). They resolve once versions are normalized,
but they would fail an exact-match lookup.

### 5.2 Associations to a different collection (839 entries, 228 collections)

These are lineage links already in CMR that the ATBD-derived CSVs missed. **They're the main candidates to add to the
lineage CSVs.**

- By type: `INPUT` 730, `PARENT` 52, `SCIENCE ASSOCIATED` 27, `RELATED` 14, `DEPENDENT` 11, `CHILD` 5.
- 811 are in LARC_CLOUD. The most-linked targets are `MICNFG` (132), `MIANCAGP` (54), `MIANCARP` (47), `MIANLDBM` (33), `MI1B2T` (32), `MI1B2E` (31).
- 454 of the 839 have an `entryId` that doesn't resolve to a CMR collection: the
  target isn't in CMR, or the ID has a version inside it or is free text (section 6). Adding those to the CSVs as-is
  would produce links that can't be resolved.

---

## 6. CMR Association Data Quality

All 1,685 association entries on the 730 resolved records were checked:

| Check | Entries | Examples |
|---|---:|---|
| `entryId` not in CMR | 445 (103 distinct targets) | `MICNFG` (132), `MIANLDBM` (33), `MIANCSSC` (24), likely typos `ASF_ECMWF_TROPO`, `PREFIRE_SAT1-2B-SFC`, `PREFIRE_SAT2-2B-SFC`, `SAGE2_V6.20_AEROSOL_O3_NO2_H2O_BINARY`, `SENIINEL-1C_SLC`, `SENIINEL-1D_SLC` |
| Version inside `entryId` | 8 | `VNP46A1_2`, `VNP46A1_1` (×3), `VJ146A2_2`, `CER_CCCM_Aqua-FM3-MODIS-CAL-CS_RelD1`, `CER_SSF_Aqua-FM3-MODIS_Edition4A` |
| Leading/trailing whitespace in `entryId` | 5 | `"OPERA_L2_CSLC-S1-STATIC_V1 "`, `"MOD09A1N "`, `"CMRMIAAE "` (×2), `" TEMPO_CLDO4_L3_NRT"` |
| Missing `type` | 4 | `Global_Landslide_Exposure_Maps`, `Global_Landslide_Nowcast` (×2), `glmmth` |
| Free-text `entryId` | 1 | `"OPeNDAP data access for CAL_IIR_L2_Track-Standard-V4-20"` in `CAL_IIR_L2_Track-Standard-V4-51` |
| `type` not in the UMM-C enum | 0 |  |
| Duplicate entries | 0 |  |

Specific errors worth reporting to the DAACs:
- **ASF `OPERA_L4_TROPO-ZENITH_V1`:** `ASF_ECMWF_TROPO` is not in CMR, probably `ASF_ECMWF_TROP`. The CSV has the correct name.
- **ASF `SENTINEL-1_BURSTS`:** `SENIINEL-1C_SLC` is not in CMR, probably `SENTINEL-1C_SLC`; `SENIINEL-1D_SLC` is not in CMR, probably `SENTINEL-1D_SLC`. The CSV has the correct name.
- **LARC_CLOUD `PREFIRE_SAT1_3-SFC-SORTED-ALLSKY`:** `PREFIRE_SAT1-2B-SFC` is not in CMR, probably `PREFIRE_SAT1_2B-SFC`. The CSV has the correct name.
- **LARC_CLOUD `PREFIRE_SAT2_3-SFC-SORTED-ALLSKY`:** `PREFIRE_SAT2-2B-SFC` is not in CMR, probably `PREFIRE_SAT2_2B-SFC`. The CSV has the correct name.
- **LARC_CLOUD `SAGE2_AEROSOL_O3_NO2_H2O_BINARY`:** `SAGE2_V6.20_AEROSOL_O3_NO2_H2O_BINARY` is not in CMR, probably `SAGE2_AEROSOL_O3_NO2_H2O_BINARY`.
- **LARC_CLOUD `CAL_IIR_L2_Track-Standard-V4-51`:** the `PARENT` entry ID is a description, not a short name.
- **LANCEMODIS:** entry IDs with the version inside them (`VJ146A2_2`, `VNP46A1_1`, `VNP46A1_2`) should be split into short name + `Version`.
- **LARC_CLOUD:** entry IDs with the version inside them (`CER_CCCM_Aqua-FM3-MODIS-CAL-CS_RelD1`, `CER_SSF_Aqua-FM3-MODIS_Edition4A`, `DSCOVR_EPIC_L2_COMPOSITE_01`) should be split into short name + `Version`.

"Not in CMR" means that CMR GraphQL returned no collection with that short name under any provider. During development
a sample (e.g. `MICNFG`, `AIRMISR`, `ASF_ECMWF_TROPO`) was confirmed against the CMR Search API
(`collections.json?short_name=`), which returns nothing for them either.

---

## 7. Recommendations

**For the lineage CSV generation skill (`Collection_Lineage_via_ATBD_Skill`)**
1. Use CMR's type convention: upstream processing inputs = `INPUT`, downstream products = `DEPENDENT`. Keep
   `PARENT`/`CHILD` for version succession. This alone would resolve most of the 178 type differences.
2. Check every `related_shortname` against CMR before writing it. That would drop or fix the 8
   non-existent names.
3. Merge in the 385 CMR links to other collections whose target exists in CMR
   (section 5.2). Leave out, or first correct, the 454 whose `entryId` doesn't resolve.

**For the DAACs / CMR metadata**
1. Fix the likely typos and the free-text entry IDs listed in section 6.
2. Trim whitespace from entry IDs, and split versions out of entry IDs into the `Version` field.
3. Add `Type` to the 4 untyped associations.
4. Review the 217 version links to versions that no longer exist, and the
   146 GES_DISC self-loop placeholders.
5. Review the 103 association targets that public CMR doesn't have (most-linked:
   `MICNFG`, `MIANLDBM`, `MIANCSSC`).

---

## 8. Reproducing the Results

Run every cell of `lineage/notebooks/lineage_association_cross_check.ipynb`. It uses the `uv_metadata` environment
(Python 3.14, `requests`) and needs no authentication. Step 9 of the notebook regenerates this report.
- `DRY_RUN = True` computes everything without rewriting the CSVs.
- `RUN_BENCHMARK = False` skips the query-strategy timing.
- `WRITE_REPORT = False` skips writing this report.

Results reflect CMR at the retrieval time shown at the top of this report.
